<a href="https://colab.research.google.com/github/aldo02032004/naufaldo.github.io/blob/main/Linear_regression_forecast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CELL 1 — PENGATURAN (hanya cell ini yang perlu diubah)


In [1]:
LINK_DRIVE = [   # BISA DIUBAH: satu link, atau beberapa link dipisah koma
    'https://docs.google.com/spreadsheets/d/1dYpzXZuAG7TPQjCtspagX-PqFiIh-HRG/edit',
    'https://docs.google.com/spreadsheets/d/1RBAQovVNUXinv3sgQNvDS1iMlciTL1vN/edit',
    'https://docs.google.com/spreadsheets/d/1ugTVwpxj6KQsZiI1dneQ-mAX2vTBVSTX/edit',
]
NAMA_OBJEK = 'Prabowo'   # BISA DIUBAH: nama objek penelitian, muncul di judul grafik

# CELL 2 — Pasang library GREAT (cukup dijalankan)

In [2]:
import subprocess, sys
from google.colab import userdata

try:
    token = userdata.get('GH_TOKEN')
except Exception:
    raise SystemExit('❌ Secret GH_TOKEN belum ada. Buka ikon 🔑 di sidebar kiri, tambahkan GH_TOKEN, '
                     'lalu nyalakan "Notebook access".')

paket = f'great[all] @ git+https://{token}@github.com/azmkto/GREAT-Tools.git'
hasil = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', paket], capture_output=True, text=True)
if hasil.returncode:
    print(hasil.stderr.replace(token, '***')[-2000:])   # token tidak pernah ditampilkan
    raise SystemExit('❌ Pemasangan gagal. Cek apakah token masih berlaku dan punya akses ke repo GREAT-Tools.')
del token
print('✅ Library GREAT terpasang')

✅ Library GREAT terpasang


# CELL 3 — Siapkan library dan angka pengaturan (cukup dijalankan)


In [9]:
import re
import time

import numpy as np
import pandas as pd
import ipywidgets as w
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.transforms import offset_copy
from IPython.display import display
from scipy.special import stdtrit          # pengali distribusi t (versi cepat)

from great import sent_class, sent_colors, validate_export   # urutan & warna sentimen dari library GREAT
from great.viz import prep
from great.viz.overview import _panel_sentiment_trend        # panel tren yang sama dengan laporan bulanan
from great.viz.style import apply_style

# ---- Angka-angka di bawah ini BOLEH DIUBAH kalau perlu ----
TINGKAT_YAKIN = 0.80      # lebar area bayangan: 0.80 = angka asli masuk area kira-kira 4 dari 5 hari (makin besar, makin lebar)
MAX_PREDIKSI = 14         # paling jauh berapa hari ke depan yang boleh diprediksi
AWAL_BELAJAR = 7          # posisi awal slider "Hari untuk belajar"
AWAL_PREDIKSI = 3         # posisi awal slider "Hari yang diprediksi"
UKURAN_GRAFIK = (30, 8)   # (lebar, tinggi) grafik
DPI_GRAFIK = 100          # ketajaman grafik: makin besar makin tajam, tapi makin lambat saat slider digeser
WARNA_KETERANGAN = 'grey'     # warna contoh garis putus-putus dan area bayangan di keterangan grafik
WARNA_DIPELAJARI = 'orange'   # warna kotak hari-hari yang dipelajari
KEKUATAN_TAHAN = (1, 10, 100)   # pilihan seberapa kuat "Regresi pintar" menahan perbedaan antar hari (dicoba semua)
TAHAN_AWAL = 10                 # kekuatan tahan yang dipakai kalau data terlalu sedikit untuk memilih

# ---- Di bawah ini sebaiknya tidak diubah ----
MIN_HARI = 2              # data minimal supaya bisa menarik tren
OTOMATIS = 'Otomatis (paling akurat)'
PINTAR = 'Regresi pintar'
# Urutan dari yang paling sederhana. Kalau skor dua metode sama, yang lebih sederhana yang dipilih.
METODE = ['Sama dengan hari terakhir', 'Rata-rata terbaru', 'Sama dengan minggu lalu',
          'Garis lurus', 'Pertumbuhan persen', PINTAR]
MIN_HARI_METODE = {'Sama dengan minggu lalu': 14, PINTAR: 7}   # data minimal (hari) sebelum metode ini ditawarkan

print('✅ Library dan pengaturan siap')

✅ Library dan pengaturan siap


# CELL 4 — Rumus-rumus prediksi (cukup dijalankan)


In [4]:
class TidakBisaDipakai(Exception):
    """Metode tidak bisa dipakai untuk pengaturan ini. Pesannya ditulis dalam bahasa sehari-hari."""


def pengali_t(derajat_bebas, level=TINGKAT_YAKIN):
    """Pengali lebar area bayangan (distribusi t). Makin sedikit data, makin besar pengalinya."""
    return float(stdtrit(derajat_bebas, 0.5 + level / 2))


def link_ke_xlsx(url):
    """Ubah link Google Sheets / Google Drive menjadi link unduhan file xlsx."""
    cocok = re.search(r'/spreadsheets/d/([\w-]+)', url)
    if cocok:
        return f'https://docs.google.com/spreadsheets/d/{cocok.group(1)}/export?format=xlsx'
    cocok = re.search(r'/file/d/([\w-]+)', url) or re.search(r'[?&]id=([\w-]+)', url)
    if cocok:
        return f'https://drive.google.com/uc?export=download&id={cocok.group(1)}'
    raise SystemExit('❌ LINK_DRIVE bukan link Google Drive / Sheets. Salin ulang dari tombol Share.')


def hitung_harian(df1, dasar):
    """Tabel hari x sentimen. Postingan = jumlah postingan per hari, Orang = jumlah akun berbeda per hari.
    Hari tanpa postingan diisi 0."""
    kelompok = df1.groupby([pd.Grouper(key='Date', freq='D'), 'Sentiment'])
    harian = kelompok.size() if dasar == 'Postingan' else kelompok['Author'].nunique()
    return (harian.unstack(fill_value=0)
                  .reindex(columns=sent_class, fill_value=0)
                  .asfreq('D', fill_value=0)
                  .astype(float))


def metode_tersedia(n):
    """Metode yang datanya cukup untuk ditawarkan."""
    return [m for m in METODE if n >= MIN_HARI_METODE.get(m, 0)]


def kolom_kalender(hari_ke):
    """Untuk setiap hari: [akhir pekan?, Senin?, Selasa?, ..., Minggu?] berisi 1 (ya) atau 0 (tidak).
    hari_ke: 0 = Senin ... 6 = Minggu."""
    return np.column_stack([hari_ke >= 5, hari_ke[:, None] == np.arange(7)]).astype(float)


# ---- Regresi pintar ----
# Angka hari ini (dalam skala log) ditebak dari:
#   angka kemarin + efek akhir pekan + koreksi kecil per hari (Senin..Minggu).
# Koreksi per hari "ditahan" supaya hanya dipakai kalau datanya jelas menunjukkannya.
_simpanan_regresi = {}


def latih_pintar(Z, hari_ke, tahan, nama):
    """Latih Regresi pintar. Hasil disimpan, jadi data yang sama tidak dihitung dua kali."""
    kunci = (Z.shape, Z.tobytes(), int(hari_ke[0]), tahan)
    if kunci not in _simpanan_regresi:
        try:
            _simpanan_regresi[kunci] = _latih_pintar(Z, hari_ke, tahan, nama)
        except TidakBisaDipakai as alasan:
            _simpanan_regresi[kunci] = alasan
    hasil = _simpanan_regresi[kunci]
    if isinstance(hasil, TidakBisaDipakai):
        raise hasil
    return hasil


def _latih_pintar(Z, hari_ke, tahan, nama):
    """Z = log(1 + angka harian), baris = hari yang dipelajari, kolom = sentimen."""
    L = len(Z)
    m = L - 1                                          # hari pertama tidak punya "kemarin"
    kalender = kolom_kalender(hari_ke[1:])
    akhir_pekan = kalender[:, 0]
    if akhir_pekan.all() or not akhir_pekan.any():
        raise TidakBisaDipakai('hari yang dipelajari harus berisi minimal satu hari kerja dan satu hari akhir pekan')
    penahan = np.diag([0.0, 0.0, 0.0] + [float(tahan)] * 7)   # hanya 7 koreksi per hari yang ditahan

    hasil = []
    for j, sentimen in enumerate(nama):
        X = np.column_stack([np.ones(m), Z[:-1, j], kalender])   # [1, kemarin, akhir pekan, Senin..Minggu]
        z = Z[1:, j]
        if np.linalg.matrix_rank(X[:, :3]) < 3:
            raise TidakBisaDipakai(f'angka {sentimen} hampir tidak berubah di hari yang dipelajari, '
                                   'jadi tidak ada yang bisa dipelajari')
        XtX = X.T @ X
        # Regresi linear dengan penahan, sekali hitung: kolom pertama = koefisien, sisanya untuk "jumlah angka efektif".
        solusi = np.linalg.solve(XtX + penahan, np.column_stack([X.T @ z, XtX]))
        koef = solusi[:, 0]                            # [dasar, efek kemarin, efek akhir pekan, koreksi Senin..Minggu]
        derajat_bebas = m - np.trace(solusi[:, 1:])
        if derajat_bebas < 2:
            raise TidakBisaDipakai(f'"Hari untuk belajar" perlu lebih banyak (sekarang {L}, coba 7 atau lebih)')
        if abs(koef[1]) >= 1:
            raise TidakBisaDipakai(f'angka {sentimen} terus lari ke satu arah di hari yang dipelajari, '
                                   'jadi modelnya tidak stabil')
        cocok = X @ koef
        hasil.append({'koef': koef, 'dof': derajat_bebas, 'cocok': cocok,
                      'sigma': np.sqrt(((z - cocok) ** 2).sum() / derajat_bebas)})
    return hasil


def ramal(Y, hari_ke_awal, L, H, metode, tahan=TAHAN_AWAL, nama=sent_class, level=TINGKAT_YAKIN):
    """Inti prediksi, memakai array angka biasa supaya cepat.
      Y             : angka harian sampai "hari ini" (baris = hari, kolom = sentimen)
      hari_ke_awal  : hari apa baris pertama Y (0 = Senin)
      L, H          : berapa hari terakhir dipelajari, berapa hari ke depan diprediksi
    Hasil: (tengah, bawah, atas), masing-masing L + H baris. Bawah/atas = area bayangan (kosong kalau tidak bisa dihitung)."""
    Yb = Y[-L:]                                        # hari-hari yang dipelajari
    tengah = np.full((L + H, Y.shape[1]), np.nan)
    bawah, atas = tengah.copy(), tengah.copy()
    langkah = np.arange(1, H + 1, dtype=float)[:, None]    # 1, 2, ..., H hari ke depan

    if metode in ('Garis lurus', 'Pertumbuhan persen'):
        Yf = np.log1p(Yb) if metode == 'Pertumbuhan persen' else Yb
        t0 = np.arange(L, dtype=float)
        X = np.column_stack([np.ones(L), t0])
        b, a = np.linalg.lstsq(X, Yf, rcond=None)[0]       # regresi linear: y = a * hari + b (per sentimen)
        t = np.arange(L + H, dtype=float)[:, None]
        tengah = a * t + b
        if L >= 3:
            s = np.sqrt(((Yf - (a * t0[:, None] + b)) ** 2).sum(axis=0) / (L - 2))
            se = s * np.sqrt(1 + 1 / L + (t - t0.mean()) ** 2 / ((t0 - t0.mean()) ** 2).sum())
            q = pengali_t(L - 2, level)
            bawah, atas = tengah - q * se, tengah + q * se
        if metode == 'Pertumbuhan persen':
            tengah, bawah, atas = np.expm1(tengah), np.expm1(bawah), np.expm1(atas)

    elif metode == 'Rata-rata terbaru':
        rata = Yb.mean(axis=0)
        tengah[:] = rata
        if L >= 2:
            se = Yb.std(axis=0, ddof=1) * np.sqrt(1 + 1 / L)
            q = pengali_t(L - 1, level)
            bawah[L:], atas[L:] = rata - q * se, rata + q * se

    elif metode == 'Sama dengan hari terakhir':
        tengah[L - 1:] = Yb[-1]
        if L >= 3:
            se = np.diff(Yb, axis=0).std(axis=0, ddof=1) * np.sqrt(langkah)
            q = pengali_t(L - 2, level)
            bawah[L:], atas[L:] = Yb[-1] - q * se, Yb[-1] + q * se

    elif metode == 'Sama dengan minggu lalu':
        tengah[L - 1] = Yb[-1]
        tengah[L:] = Y[-7:][np.arange(H) % 7]

    elif metode == PINTAR:
        n = len(Y)
        hari_ke = (hari_ke_awal + np.arange(n + H)) % 7     # hari apa untuk semua baris + hari ke depan
        Z = np.log1p(Yb)
        hasil = latih_pintar(Z, hari_ke[n - L:n], tahan, nama)
        kalender_depan = kolom_kalender(hari_ke[n:])
        for j, f in enumerate(hasil):
            dasar, phi, sisanya = f['koef'][0], f['koef'][1], f['koef'][2:]
            efek_hari = kalender_depan @ sisanya              # efek akhir pekan + koreksi hari untuk tiap hari depan
            z, zf = Z[-1, j], np.empty(H)                      # mulai dari angka asli hari terakhir
            for h in range(H):
                z = dasar + phi * z + efek_hari[h]             # tebakan besok dipakai untuk menebak lusa, dst.
                zf[h] = z
            se = f['sigma'] * np.sqrt(np.cumsum(phi ** (2 * np.arange(H))))
            q = pengali_t(f['dof'], level)
            tengah[0, j] = Z[0, j]                             # hari pertama tidak punya tebakan: pakai angka asli
            tengah[1:L, j] = f['cocok']
            tengah[L:, j] = zf
            bawah[L:, j], atas[L:, j] = zf - q * se, zf + q * se
        tengah, bawah, atas = np.expm1(tengah), np.expm1(bawah), np.expm1(atas)

    else:
        raise ValueError(metode)

    return np.clip(tengah, 0, None), np.clip(bawah, 0, None), np.clip(atas, 0, None)   # jumlah tidak mungkin minus


def ramal_tabel(y, L, H, metode, tahan=TAHAN_AWAL):
    """Sama seperti ramal(), tapi hasilnya tabel bertanggal (untuk grafik)."""
    hasil = ramal(y.to_numpy(float), y.index[0].dayofweek, L, H, metode, tahan, list(y.columns))
    tanggal = y.index[-L:].append(pd.date_range(y.index[-1], periods=H + 1, freq='D')[1:])
    return [pd.DataFrame(v, index=tanggal, columns=y.columns) for v in hasil]


def uji_akurasi(y, L, H, daftar_metode):
    """Uji ke masa lalu: anggap hari-hari sebelumnya adalah "hari ini", prediksi H hari berikutnya,
    lalu bandingkan dengan angka aslinya. Semua metode diuji di hari yang sama.
    Regresi pintar dicoba dengan setiap kekuatan tahan, lalu yang paling akurat dipakai
    (catatan: karena dipilih dari uji yang sama, skornya sedikit terlalu optimis).
      akurasi = 100% dikurangi rata-rata meleset (dibanding angka asli)
      cakupan = seberapa sering angka asli masuk area bayangan"""
    Y = y.to_numpy(float)
    hari_ke_awal = y.index[0].dayofweek
    varian = [(m, t) for m in daftar_metode for t in (KEKUATAN_TAHAN if m == PINTAR else (TAHAN_AWAL,))]
    mulai = max(L, 7) if 'Sama dengan minggu lalu' in daftar_metode else L
    meleset = {v: 0.0 for v in varian}
    masuk = {v: [] for v in varian}
    dilewati = {}
    total_asli, jumlah_uji = 0.0, 0
    for akhir in range(mulai, len(Y) - H + 1):          # setiap "hari ini" pura-pura
        riwayat, asli = Y[:akhir], Y[akhir:akhir + H]
        total_asli += asli.sum()
        jumlah_uji += 1
        for v in varian:
            if v in dilewati:
                continue
            m, t = v
            try:
                tengah, bawah, atas = (x[L:] for x in ramal(riwayat, hari_ke_awal, L, H, m, t, list(y.columns)))
            except TidakBisaDipakai as alasan:
                dilewati[v] = f'pada {y.index[akhir - 1]:%d %b %Y} {alasan}'
                continue
            meleset[v] += np.abs(tengah - asli).sum()
            if not np.isnan(bawah).all():
                masuk[v].append(((asli >= bawah) & (asli <= atas)).mean())
    if jumlah_uji == 0 or total_asli == 0:
        return None

    akurasi = {v: max(0.0, 1 - meleset[v] / total_asli) for v in varian if v not in dilewati}
    tahan = TAHAN_AWAL
    tahan_ok = [t for t in KEKUATAN_TAHAN if (PINTAR, t) in akurasi]
    if PINTAR in daftar_metode and jumlah_uji >= 3 and tahan_ok:
        tahan = max(tahan_ok, key=lambda t: akurasi[(PINTAR, t)])
    pilih = {m: (m, tahan if m == PINTAR else TAHAN_AWAL) for m in daftar_metode}
    return {'jumlah_uji': jumlah_uji, 'tahan': tahan,
            'akurasi': {m: akurasi[v] for m, v in pilih.items() if v in akurasi},
            'cakupan': {m: (float(np.mean(masuk[v])) if masuk[v] else None)
                        for m, v in pilih.items() if v in akurasi},
            'dilewati': {m: dilewati[v] for m, v in pilih.items() if v in dilewati}}


print(f'✅ {len(METODE)} metode prediksi siap: ' + ', '.join(METODE))

✅ 6 metode prediksi siap: Sama dengan hari terakhir, Rata-rata terbaru, Sama dengan minggu lalu, Garis lurus, Pertumbuhan persen, Regresi pintar


# CELL 5 — Ambil data dari Google Drive (cukup dijalankan)

In [10]:
daftar_link = [x.strip() for x in ([LINK_DRIVE] if isinstance(LINK_DRIVE, str) else LINK_DRIVE) if x.strip()]
if not daftar_link:
    raise SystemExit('❌ LINK_DRIVE kosong. Tempel minimal satu link Google Drive / Sheets di Cell 1.')

bagian = []
for i, link in enumerate(daftar_link, 1):
    label = f'link {i} dari {len(daftar_link)}'
    try:
        tabel = pd.read_excel(link_ke_xlsx(link), header=1)
    except SystemExit:
        raise
    except Exception as e:
        raise SystemExit(f'❌ {label} tidak bisa dibuka ({e}).\n'
                         'Pastikan file Drive dibagikan sebagai "Anyone with the link".')
    try:
        bagian.append(validate_export(tabel))
    except ValueError as e:
        raise SystemExit(f'❌ File di {label} formatnya tidak sesuai: {e}')
    print(f'   {label}: {len(tabel):,} baris')

df = pd.concat(bagian, ignore_index=True)
# File yang tanggalnya tumpang tindih berisi postingan yang sama dua kali. Kolom 'No' hanya nomor baris, jadi diabaikan.
df['Date'] = pd.to_datetime(df['Date'], format='ISO8601')
sebelum = len(df)
df = df.drop_duplicates(subset=[c for c in df.columns if c != 'No']).reset_index(drop=True)
if len(df) < sebelum:
    print(f'   {sebelum - len(df):,} baris ganda (ada di lebih dari satu file) dibuang')

df1, tanggal_awal, tanggal_akhir = prep.prepare_data(df)
n_hari = (df1['Date'].max() - df1['Date'].min()).days + 1
print(f'✅ {len(df1):,} baris siap | {tanggal_awal} - {tanggal_akhir} | {n_hari} hari')
if n_hari < MIN_HARI:
    raise SystemExit(f'❌ Data cuma {n_hari} hari. Untuk menarik tren butuh minimal {MIN_HARI} hari.')

display(df1[['Date', 'Media', 'Sentiment', 'Author']].head())

   link 1 dari 3: 150,000 baris
   link 2 dari 3: 150,000 baris
   link 3 dari 3: 18,793 baris
   50,213 baris ganda (ada di lebih dari satu file) dibuang
✅ 268,580 baris siap | 27 Aug 2026 - 25 Sep 2026 | 30 hari


,Date,Media,Sentiment,Author
268579,2026-08-27,News,positive,kabarterkini24.com
260830,2026-08-27,Instagram,positive,NaN
260829,2026-08-27,Twitter,positive,@EKubang
260828,2026-08-27,Youtube,positive,GARUDA TV
260827,2026-08-27,News,neutral,dataindonesia.id


# CELL 6 — Hitung jumlah per hari (cukup dijalankan)


In [6]:
harian = {dasar: hitung_harian(df1, dasar) for dasar in ('Postingan', 'Orang')}
print('Jumlah POSTINGAN per hari (10 hari terakhir):')
display(harian['Postingan'].tail(10).astype(int))
print('Jumlah ORANG (akun berbeda) per hari (10 hari terakhir):')
display(harian['Orang'].tail(10).astype(int))

Jumlah POSTINGAN per hari (10 hari terakhir):


Sentiment,positive,negative,neutral
Date,,,
2026-09-16,3502,3868,604
2026-09-17,4165,3684,346
2026-09-18,3920,3420,781
2026-09-19,5158,3890,1671
2026-09-20,3290,4855,539
2026-09-21,2820,3288,665
2026-09-22,3161,5650,768
2026-09-23,4080,4882,1171
2026-09-24,3633,4891,917


Jumlah ORANG (akun berbeda) per hari (10 hari terakhir):


Sentiment,positive,negative,neutral
Date,,,
2026-09-16,1757,1660,346
2026-09-17,2256,1591,234
2026-09-18,2374,1547,384
2026-09-19,2994,1857,479
2026-09-20,2001,3365,201
2026-09-21,1754,1652,259
2026-09-22,1771,4064,434
2026-09-23,2632,3361,558
2026-09-24,2379,3559,375


# CELL 7 — Uji akurasi dengan pengaturan awal (cukup dijalankan)

In [11]:
_simpanan = {}

def simpan(kunci, hitung):
    """Hitung sekali saja untuk setiap kunci, supaya slider tetap cepat saat digeser bolak-balik."""
    if kunci not in _simpanan:
        _simpanan[kunci] = hitung()
    return _simpanan[kunci]


def ambil_uji(dasar, L, H):
    y = harian[dasar]
    return simpan(('uji', dasar, L, H), lambda: uji_akurasi(y, L, H, metode_tersedia(len(y))))


def ambil_ramalan(dasar, L, H, metode, tahan):
    return simpan(('ramal', dasar, L, H, metode, tahan), lambda: ramal_tabel(harian[dasar], L, H, metode, tahan))


L_awal, H_awal = min(AWAL_BELAJAR, n_hari), min(AWAL_PREDIKSI, n_hari)
jam_mulai = time.perf_counter()
uji_awal = ambil_uji('Postingan', L_awal, H_awal)
print(f'Uji akurasi (belajar {L_awal} hari, prediksi {H_awal} hari) selesai dalam '
      f'{time.perf_counter() - jam_mulai:.1f} detik')
if uji_awal is None:
    print('Data belum cukup untuk diuji dengan pengaturan ini.')
else:
    print(f'Diuji {uji_awal["jumlah_uji"]} kali. Akurasi tiap metode:')
    display(pd.Series(uji_awal['akurasi']).sort_values(ascending=False).map('{:.0%}'.format).to_frame('Akurasi'))

Uji akurasi (belajar 7 hari, prediksi 3 hari) selesai dalam 0.0 detik
Diuji 21 kali. Akurasi tiap metode:


,Akurasi
Rata-rata terbaru,69%
Sama dengan hari terakhir,66%
Pertumbuhan persen,64%
Garis lurus,62%
Sama dengan minggu lalu,57%


# CELL 8 — Grafik prediksi dengan pilihan (cukup dijalankan, lalu atur pilihannya)


In [12]:
apply_style(dpi=DPI_GRAFIK)
lebar_label = {'description_width': '160px'}

# Dua pilihan utama.
dasar = w.ToggleButtons(options=['Postingan', 'Orang'], description='Yang dihitung', style=lebar_label,
                        tooltips=['Jumlah postingan per hari', 'Jumlah akun berbeda yang memposting per hari'])
prediksi = w.IntSlider(min=1, max=min(MAX_PREDIKSI, n_hari), value=min(AWAL_PREDIKSI, n_hari),
                       description='Hari yang diprediksi', style=lebar_label)

# Pilihan tambahan, disimpan di kotak "Pengaturan lanjutan" (tertutup sampai dibuka).
metode = w.Dropdown(options=[OTOMATIS] + metode_tersedia(n_hari), value=OTOMATIS, description='Metode',
                    style=lebar_label)
belajar = w.IntSlider(min=MIN_HARI, max=n_hari, value=min(AWAL_BELAJAR, n_hari),
                      description='Hari untuk belajar', style=lebar_label)
penjelasan = w.HTML('<b>Metode</b>: biarkan Otomatis, nanti dipilih yang paling akurat.<br>'
                    '<b>Hari untuk belajar</b>: berapa hari terakhir yang dilihat. '
                    'Lebih banyak = lebih tenang, lebih sedikit = lebih cepat bereaksi.')
lanjutan = w.Accordion(children=[w.VBox([w.HBox([metode, belajar]), penjelasan])], selected_index=None)
lanjutan.set_title(0, 'Pengaturan lanjutan (tidak wajib)')


def teks_hasil(uji, dipakai, dari_otomatis, n, L, H, catatan=None):
    """Ringkasan singkat di bawah grafik."""
    baris = [catatan] if catatan else []
    baris.append(f'Metode: {dipakai}' + (' (dipilih otomatis)' if dari_otomatis else ''))
    if uji is None:
        baris.append('Akurasi: belum bisa dihitung, datanya kurang. Coba kecilkan "Hari untuk belajar".')
    else:
        akurasi = uji['akurasi']
        if dipakai in akurasi:
            baris.append(f'Akurasi di hari-hari sebelumnya: {akurasi[dipakai]:.0%} (diuji {uji["jumlah_uji"]} kali)')
        cakupan = uji['cakupan'].get(dipakai)
        if cakupan is not None:
            baris.append(f'Angka asli masuk area bayangan: {cakupan:.0%} dari waktu')
        urutan = sorted(akurasi, key=akurasi.get, reverse=True)
        baris.append('Semua metode: ' + ' · '.join(f'{m} {akurasi[m]:.0%}' for m in urutan))
    if n < 7:
        baris.append('⚠️ Data kurang dari seminggu, jadi hasilnya masih kasar.')
    if H > L:
        baris.append('⚠️ Memprediksi lebih jauh dari jumlah hari yang dipelajari, kemungkinan meleset lebih besar.')
    return '\n'.join(baris)


def gambar(dasar_v, metode_v, belajar_v, prediksi_v):
    y = harian[dasar_v]
    n = len(y)
    L, H = min(belajar_v, n), min(prediksi_v, n)

    uji = ambil_uji(dasar_v, L, H)
    tahan = uji['tahan'] if uji else TAHAN_AWAL
    dari_otomatis = metode_v == OTOMATIS
    if dari_otomatis:
        dipakai = max(uji['akurasi'], key=uji['akurasi'].get) if uji else 'Rata-rata terbaru'
    else:
        dipakai = metode_v

    catatan = None
    try:
        tengah, bawah, atas = ambil_ramalan(dasar_v, L, H, dipakai, tahan)
    except TidakBisaDipakai as alasan:
        catatan = f'"{dipakai}" tidak bisa dipakai di pengaturan ini ({alasan}), jadi diganti "Rata-rata terbaru".'
        dipakai = 'Rata-rata terbaru'
        tengah, bawah, atas = ambil_ramalan(dasar_v, L, H, dipakai, tahan)
    depan, depan_bawah, depan_atas = (x.iloc[L:] for x in (tengah, bawah, atas))

    fig, ax = plt.subplots(figsize=UKURAN_GRAFIK)
    _panel_sentiment_trend(ax, y, sent_class, sent_colors, NAMA_OBJEK, tanggal_awal, tanggal_akhir, interval=1)
    ax.set_title(f'Sentiment of {NAMA_OBJEK}\n{y.index[0]:%d %b %Y} - {y.index[-1]:%d %b %Y}',
                 fontsize=20, fontweight='bold', pad=20)

    # Garis putus-putus = prediksi, mulai dari titik asli terakhir supaya tersambung.
    garis = pd.concat([y.iloc[[-1]], depan])
    for s in sent_class:
        warna = sent_colors[s]
        ax.plot(garis.index, garis[s], color=warna, linewidth=3, linestyle='--', marker='o', ms=3.01, alpha=0.8)
        ax.fill_between(depan.index, depan_bawah[s], depan_atas[s], color=warna, alpha=0.12, linewidth=0)

    # Angka di ujung garis. Kalau angka bertumpuk, yang di atas digeser naik.
    y_min, y_max = ax.get_ylim()
    jarak, sebelumnya = 0.05 * (y_max - y_min), -np.inf
    x_ujung = mdates.date2num(depan.index[-1])
    geser = offset_copy(ax.transData, fig=fig, x=8, units='points')
    for s in sorted(sent_class, key=lambda s: depan[s].iloc[-1]):
        akhir, rendah, tinggi = depan[s].iloc[-1], depan_bawah[s].iloc[-1], depan_atas[s].iloc[-1]
        teks = f'{akhir:,.0f}' if np.isnan(rendah) else f'{akhir:,.0f} ({rendah:,.0f}–{tinggi:,.0f})'
        sebelumnya = max(akhir, sebelumnya + jarak)
        ax.annotate(teks, (x_ujung, akhir), xytext=(x_ujung, sebelumnya), textcoords=geser,
                    va='center', fontsize=12, color=sent_colors[s], fontweight='bold')
    ax.axvspan(y.index[-L], y.index[-1], color=WARNA_DIPELAJARI, alpha=0.12)

    # Keterangan grafik: warna sentimen + arti garis putus-putus, area bayangan, dan kotak hari yang dipelajari.
    simbol, nama = ax.get_legend_handles_labels()
    simbol += [Line2D([], [], color=WARNA_KETERANGAN, linewidth=3, linestyle='--'),
               Patch(color=WARNA_KETERANGAN, alpha=0.25),
               Patch(color=WARNA_DIPELAJARI, alpha=0.25)]
    nama += ['Prediksi', f'Rentang wajar prediksi ({TINGKAT_YAKIN:.0%} yakin)', f'Hari yang dipelajari ({L} hari)']
    ax.legend(simbol, nama, fontsize=12, loc='upper center', bbox_to_anchor=(0.5, 1.05),
              frameon=False, ncol=len(nama))

    semua_tanggal = y.index.append(depan.index)
    ax.set_xticks(semua_tanggal[::max(1, len(semua_tanggal) // 15)])   # paling banyak ~15 tanggal di sumbu bawah
    ax.set_xlabel('Tanggal', fontsize=18, fontweight='semibold')
    ax.set_ylabel(f'{dasar_v} per hari', fontsize=18, fontweight='semibold')
    plt.tight_layout()
    plt.show()

    print(teks_hasil(uji, dipakai, dari_otomatis, n, L, H, catatan))


keluaran = w.interactive_output(gambar, {'dasar_v': dasar, 'metode_v': metode,
                                         'belajar_v': belajar, 'prediksi_v': prediksi})
display(w.VBox([w.HBox([dasar, prediksi]), lanjutan, keluaran]))